<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSEaxoapR7GyB6FC7xp4aM-SrjhxTWTdFCmKAX3TvvFvQ&s=10" alt="Tecnológico de Costa Rica" width="160">

# 🌎 WALC 2026
### Ciencia de Datos · Introducción al Aprendizaje Automático
**Día 4 (Jueves) — No supervisado — Agrupación (K-means) y asociación (Apriori)**

**Instituto Tecnológico de Costa Rica (TEC)** · Instructor: **Efrén Jiménez Delgado**
· 📧 efjimenez@itcr.ac.cr · 🔗 [LinkedIn](https://www.linkedin.com/in/ejimenezdelgado/)

# 🗺️ K-means: agrupar los lugares de delitos de una ciudad en zonas

## 🧵 El hilo conductor (metodología CRISP-DM)

Este cuaderno sigue las seis fases de **CRISP-DM** [1] [2]: cada título de fase marca dónde estamos en el proceso.

| Fase | En este cuaderno |
|---|---|
| 🎯 Comprensión del negocio | Agrupar geográficamente los delitos para identificar zonas y puntos críticos. |
| 🔍 Comprensión de los datos | Cargamos 33 116 delitos con fecha, latitud, longitud y magnitud (coordenadas de Washington D. C.). |
| 🧹 Preparación de los datos | Convertimos el tipo "ARSON" en un código numérico y revisamos valores faltantes. |
| 🤖 Modelado | Probamos distintos k y ajustamos K-means con k = 9 sobre latitud y longitud. |
| 📊 Evaluación | Describimos cada zona: tamaño, centro, extensión. |
| 🚀 Despliegue y conclusiones | Validamos con silueta y estabilidad; interpretamos las zonas. |

> 🎯 **Analogía:** colocar chinchetas en un mapa y dividirlo en zonas, de modo que cada chincheta quede en la zona de la "base de patrullas" más cercana.

### 🗺️ Mapa del curso (cronograma Track 5 · Ciencia de datos)
| Día | Jornada | Teoría y laboratorio |
|---|---|---|
| 1 | Lunes | Predicción: regresión lineal y series de tiempo |
| 2 | Martes | Clasificación 1: árboles de decisión, bosques aleatorios y KNN |
| 3 | Miércoles | Clasificación 2: SVM, regresión logística, Naive Bayes y redes neuronales |
| 👉 **4** | Jueves | **No supervisado: agrupación y asociación** |
| 5 | Viernes | Ciencia de datos con IA generativa, ética y buenas prácticas |

# Agrupar los lugares de delitos de una ciudad (Washington D. C.)

K-means particiona *n* observaciones en *k* grupos, de modo que cada observación pertenece al grupo cuyo centroide es más cercano. Aquí lo aplicamos a las coordenadas de delitos para delimitar zonas.

> Nota: las coordenadas del conjunto de datos (≈ 38.9° N, 77.0° O) corresponden a **Washington D. C.**, no a Nueva York como decía la versión original.

### Autores
* Efrén Jiménez

---
## 🎯 Fase 1 · Comprensión del negocio

Agrupar geográficamente los delitos para identificar zonas y puntos críticos.

📚 Lectura recomendada: [3]

## Análisis del Problema

La análisis de patrones delictivos es una técnica utilizada para identificar patrones, tendencias y relaciones dentro de los datos de criminalidad. Al examinar dónde, cuándo y por qué es probable que ocurran ciertos delitos, las agencias encargadas de hacer cumplir la ley pueden desarrollar estrategias efectivas de prevención del delito y dirigir sus recursos de manera más eficiente.

La análisis de patrones delictivos generalmente implica el uso de diversas técnicas estadísticas y de minería de datos para analizar los datos de criminalidad. Esto incluye técnicas espaciales para analizar la distribución geográfica de los delitos, técnicas temporales para analizar los patrones de tiempo de los delitos y técnicas de análisis de redes para analizar las relaciones entre los delitos y otros factores relevantes. Al comprender mejor la dinámica de la actividad delictiva, las agencias encargadas de hacer cumplir la ley pueden mejorar su capacidad para prevenir y resolver delitos.

### 🧪 Teoría rápida: ¿cómo agrupa K-means?

En el aprendizaje **no supervisado** no hay etiqueta que predecir: se buscan **estructuras** en los datos. K-means reparte las observaciones en *k* grupos alternando dos pasos hasta que los centroides dejan de moverse: (1) asignar cada punto al centroide más cercano y (2) recalcular cada centroide como la media de su grupo [4] [5].

![Datos sin etiquetas](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch11/images/11_01.png)
*Figura: datos sin etiquetas. [11]*

![K-means con k = 3](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch11/images/11_02.png)
*Figura: K-means con k = 3; las estrellas rojas son los centroides. [11]*

**¿Cómo elegir k?** El **método del codo** busca el *k* a partir del cual la suma de cuadrados dentro de los grupos (SSE) deja de bajar bruscamente [6]; el **coeficiente de silueta** mide qué tan bien separado queda cada punto de los demás grupos [7].

![Método del codo](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch11/images/11_03.png)
*Figura: método del codo (el codo está en k = 3). [11]*

⚠️ K-means usa **distancias**, por lo que las variables con valores grandes dominan el resultado; normalmente se **estandarizan** (y, si son muy asimétricas, se aplica logaritmo) [8].

### Librerías

In [ ]:
import warnings
warnings.filterwarnings('ignore')
#imports
#numpy,pandas,scipy, math, matplotlib
import numpy as np
import pandas as pd
import scipy
from math import sqrt
import matplotlib.pyplot as plt
import seaborn as sb

#estimators
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances_argmin_min

import plotly.graph_objs as go
import plotly.express as px

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder


---
## 🔍 Fase 2 · Comprensión de los datos

Cargamos 33 116 delitos con fecha, latitud, longitud y magnitud (coordenadas de Washington D. C.).

📚 Lectura recomendada: [3]

## Entendimiento de los Datos

El total es de 33116 registro con 4 columnas dentro del set de datos de entrenamiento



*   Fecha: la fecha y hora en que ocurrió el crimen en formato ISO 8601 (AAAA-MM-DDTHH: MM: SS.MSZ)
*   Latitud: la latitud del crimen
*   Longitud: la longitud del crimen
*   Magnitud: la magnitud del crimen

In [ ]:
import os, urllib.request
os.makedirs('sample_data', exist_ok=True)
if not os.path.exists('sample_data/crime.csv'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/namanvashistha/doctor_strange/master/crime.csv', 'sample_data/crime.csv')

In [ ]:
data = pd.read_csv('sample_data/crime.csv')
data.head()

---
## 🧹 Fase 3 · Preparación de los datos

Convertimos el tipo "ARSON" en un código numérico y revisamos valores faltantes.

In [ ]:
# 'Magnitude' mezcla números (como texto) y la palabra 'ARSON' (incendio provocado): se le asigna el código 9 y se convierte a número
data['Magnitude'] = pd.to_numeric(data['Magnitude'].replace('ARSON', '9'))
data.info()

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(data.Longitude, data.Latitude, s=2, alpha=0.3)
plt.xlabel('Longitud')
plt.ylabel('Latitud')
plt.show()

In [ ]:
print('Valores faltantes por columna:'); print(data.isna().sum())

---
## 🤖 Fase 4 · Modelado

Probamos distintos k y ajustamos K-means con k = 9 sobre latitud y longitud.

📚 Lectura recomendada: [4] [6]

## Obtener el valor K

In [ ]:
# SSE (inercia) para distintos valores de k
sse = []
k_range = range(1, 20)
for k in k_range:
    km = KMeans(n_clusters=k, max_iter=100, n_init=10, random_state=42).fit(data[['Latitude', 'Longitude']])
    sse.append(km.inertia_)

In [ ]:
# trazar gráfico de SSE vs. k
plt.plot(k_range, sse)
plt.xlabel('Número de clusters')
plt.ylabel('Score')
plt.title('Método del codo')
plt.show()

 # Modelo de Machine Learning

Una vez seleccionadas las variables para incluir en el modelo de regresión, se procede a crearlo:

In [ ]:
modelo_agrupacion = KMeans(n_clusters=9, max_iter=100, n_init=10, random_state=42)
modelo_agrupacion.fit(data[['Latitude', 'Longitude']])
data['Grupo'] = modelo_agrupacion.labels_

In [ ]:
centroides = modelo_agrupacion.cluster_centers_
pd.DataFrame(centroides, columns=['Latitude', 'Longitude']).round(4)

In [ ]:
labels = modelo_agrupacion.predict(data[['Latitude', 'Longitude']])
print(labels[:20])

---
## 📊 Fase 5 · Evaluación

Describimos cada zona: tamaño, centro, extensión.

📚 Lectura recomendada: [7]

## Evaluación

In [ ]:
data['Grupo'].value_counts().sort_index()

In [ ]:
# calcular el promedio de las columnas 'column1' y 'column2' para cada grupo en 'ventas'
data.groupby('Grupo')[['Latitude','Longitude']].mean()

In [ ]:
# Obtener el valor máximo de cada grupo para todas las columnas
data.groupby('Grupo').max()

In [ ]:
# Obtener el valor mínimo de cada grupo para todas las columnas
data.groupby('Grupo').min()

In [ ]:
# Obtener tanto el valor máximo como el mínimo de cada grupo para todas las columnas
data.groupby('Grupo').agg(['max', 'min'])

---
## 🚀 Fase 6 · Despliegue y conclusiones

Validamos con silueta y estabilidad; interpretamos las zonas.

📚 Lectura recomendada: [8]

## Resultados finales

K-means con k = 9 reparte los 33 116 delitos en nueve zonas con entre **1 419 y 5 364** delitos cada una. Cada centroide es el "centro de gravedad" de su zona (un punto crítico candidato).

### Conclusiones finales

- La silueta es casi plana (0.41–0.44 para k de 3 a 12) y la estabilidad entre semillas es moderada (≈ 0.69): **no hay una estructura natural clara**; elegir k es una decisión operativa, no estadística.
- Se agrupó solo por ubicación; `Date` y `Magnitude` no intervinieron. Zonas con más delitos no implican zonas más peligrosas sin considerar población ni gravedad.
- Siguiente paso: DBSCAN o agrupación por franja horaria y tipo de delito.

---
## 🧪 Refuerzo: ¿cuántas zonas? Silueta, estabilidad y mapa de puntos críticos

El método del codo sobre coordenadas casi nunca da un codo claro: al aumentar *k* el SSE siempre baja [6]. Para decidir cuántas zonas de delito conviene usar combinamos dos criterios: el **coeficiente de silueta** [7] y la **estabilidad** (¿obtenemos casi los mismos grupos con otra semilla?, medido con el índice de Rand ajustado). Después graficamos las zonas como un mapa de puntos con sus centroides ("puntos críticos").

In [ ]:
from sklearn.metrics import silhouette_score, adjusted_rand_score

XY = data[['Longitude', 'Latitude']]
muestra = XY.sample(5000, random_state=0)
for k in [3, 5, 7, 9, 12]:
    etq = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(muestra)
    print(f'k = {k:2d}: silueta = {silhouette_score(muestra, etq):.3f}')

# Estabilidad con k = 9: dos semillas distintas, un solo arranque cada una
a = KMeans(n_clusters=9, n_init=1, random_state=1).fit_predict(XY)
b = KMeans(n_clusters=9, n_init=1, random_state=2).fit_predict(XY)
print('Estabilidad (Rand ajustado) entre dos semillas:', round(adjusted_rand_score(a, b), 3))

# Mapa de zonas y puntos críticos
plt.figure(figsize=(7, 7))
plt.scatter(data.Longitude, data.Latitude, c=data.Grupo, cmap='tab10', s=3, alpha=0.4)
plt.scatter(centroides[:, 1], centroides[:, 0], c='black', marker='X', s=140, label='centroides')
plt.xlabel('Longitud'); plt.ylabel('Latitud'); plt.title('Zonas de delito (k = 9)'); plt.legend(); plt.show()

### ✍️ Ejercicios
1. ¿Qué *k* maximiza la silueta? ¿Es el mismo que usaría un cuerpo policial para repartir patrullas?
2. K-means supone grupos redondeados. ¿Qué pasaría con una zona alargada (una avenida)? Investigue **DBSCAN**, que agrupa por densidad [8].
3. La columna `Date` no se usó. ¿Cómo cambiarían las zonas si solo se consideran los delitos nocturnos?

---
## ✅ Conclusiones y siguientes pasos

Con k = 9 el mapa se divide en zonas de entre 1 419 y 5 364 delitos. La silueta ronda 0.41–0.44 para k entre 3 y 12 y no mejora con ningún k, y la estabilidad entre semillas es moderada (Rand ajustado ≈ 0.69): los delitos **no forman grupos naturales**, y K-means sirve aquí como una *partición del territorio* (útil para repartir recursos), no como el descubrimiento de "bandas" de delitos [8]. Para zonas de forma irregular conviene probar métodos por densidad como DBSCAN.

---
## 📚 Referencias

Fuentes adicionales para profundizar: [9], [10].

**[1]** Chapman, P., Clinton, J., Kerber, R., Khabaza, T., Reinartz, T., Shearer, C., & Wirth, R. (2000). *CRISP-DM 1.0: Step-by-step data mining guide*. SPSS Inc. https://www.the-modeling-agency.com/crisp-dm.pdf

**[2]** Wirth, R., & Hipp, J. (2000). CRISP-DM: Towards a standard process model for data mining. *Proceedings of the 4th International Conference on the Practical Applications of Knowledge Discovery and Data Mining*, 29–39.

**[3]** namanvashistha. (s. f.). *doctor_strange: crime.csv* [Conjunto de datos de delitos con fecha, latitud, longitud y magnitud]. GitHub. https://github.com/namanvashistha/doctor_strange

**[4]** MacQueen, J. (1967). Some methods for classification and analysis of multivariate observations. *Proceedings of the Fifth Berkeley Symposium on Mathematical Statistics and Probability, 1*, 281–297.

**[5]** Lloyd, S. P. (1982). Least squares quantization in PCM. *IEEE Transactions on Information Theory, 28*(2), 129–137. https://doi.org/10.1109/TIT.1982.1056489

**[6]** Thorndike, R. L. (1953). Who belongs in the family? *Psychometrika, 18*(4), 267–276. https://doi.org/10.1007/BF02289263

**[7]** Rousseeuw, P. J. (1987). Silhouettes: A graphical aid to the interpretation and validation of cluster analysis. *Journal of Computational and Applied Mathematics, 20*, 53–65. https://doi.org/10.1016/0377-0427(87)90125-7

**[8]** Jain, A. K. (2010). Data clustering: 50 years beyond K-means. *Pattern Recognition Letters, 31*(8), 651–666. https://doi.org/10.1016/j.patrec.2009.09.011

**[9]** James, G., Witten, D., Hastie, T., Tibshirani, R., & Taylor, J. (2023). *An Introduction to Statistical Learning with Applications in Python*. Springer. https://doi.org/10.1007/978-3-031-38747-0

**[10]** Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., … Duchesnay, E. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830. https://jmlr.org/papers/v12/pedregosa11a.html

**[11]** Raschka, S., & Mirjalili, V. (2019). *Python Machine Learning* (3.ª ed.). Packt Publishing. (Figuras usadas desde https://github.com/rasbt/python-machine-learning-book-3rd-edition)